In [18]:
clear_train_dir = False

DATASET = "6DMG"
num_label = 20
EPOCHS = 300
BATCH_SIZE = 64

# list of layers: [numcores; axon; neuron]
model_configs = [[13, 238, 64], [4, 208 ,64], [1, 256, 240]]
# list of config for each layer: [thres, activation_factor]
network_activation = [[0,0.7], [0,0.6], [0,0.8]]

# Path

In [19]:
# Set up base dirs
import os

ROOT_DIR = os.getcwd()

SOFT_DIR=ROOT_DIR+"/Software"
HARD_DIR=ROOT_DIR+"/Hardware"

DATASET_DIR = SOFT_DIR+"/data/processed/"
TRAIN_DIR=SOFT_DIR+"/training"
LOG_DIR=SOFT_DIR+"/log/"+DATASET

os.makedirs(LOG_DIR, exist_ok=True)

# Install

In [20]:
!pip install tensorflow
!pip install keras


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [21]:
%cd {SOFT_DIR}

if (clear_train_dir):
    !rm -rf {TRAIN_DIR}
    !unzip "training.zip"

/workspaces/SNN_framework/Software


In [22]:
%cd {TRAIN_DIR}
!pip install "./tealayers/tealayer2.0"
!pip install "./edabkutils"

/workspaces/SNN_framework/Software/training


Processing ./tealayers/tealayer2.0
  Installing build dependencies ... done
  Getting requirements to build wheel ... error
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [21 lines of output]
      Traceback (most recent call last):
        File "/usr/local/python/3.12.1/lib/python3.12/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 389, in <module>
          main()
        File "/usr/local/python/3.12.1/lib/python3.12/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        File "/usr/local/python/3.12.1/lib/python3.12/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 143, in get_requires_for_build_wheel
          return hook(config_settings)
                 ^^^^^^^^^^^^^^^^^^^^^
    

# Train

## Prepare and import package

In [23]:
from tealayer2 import Tea, AdditivePooling, tea_weight_initializer
from tensorflow.keras.layers import Flatten, Activation, Input, Lambda, Concatenate
from tensorflow.keras.losses import CategoricalFocalCrossentropy,BinaryCrossentropy, CategoricalCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical, plot_model
from tensorflow.keras import Model
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler, ModelCheckpoint

import json
import yaml
import numpy as np
import math
import tensorflow.compat.v1 as tf
from tensorflow.keras.optimizers import Adam
from sklearn.utils import class_weight

from edabkutils.modelize import auto_train_config, save_configure_json, get_configs, get_core_arrange, write_config_sim

I0000 00:00:1772882801.431197    1922 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1772882825.599862    1922 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1772882837.535481    1922 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [24]:
number_of_layers = len(model_configs)
[x_range, y_range, core_arrange] = get_core_arrange(model_configs)

print("Model configurations:", model_configs)
print("Core arrangement:", core_arrange)

Model configurations: [[13, 238, 64], [4, 208, 64], [1, 256, 240]]
Core arrangement: [[[0, 0], [1, 0], [2, 0], [3, 0], [4, 0], [5, 0], [6, 0], [7, 0], [8, 0], [9, 0], [10, 0], [11, 0], [12, 0]], [[0, 1], [1, 1], [2, 1], [3, 1]], [[0, 2]]]


In [25]:
data = np.load(DATASET_DIR+DATASET+".npz")

X_train = data["X_train"]
y_train = data["y_train"]

X_test = data["X_test"]
y_test = data["y_test"]

class_weights = class_weight.compute_class_weight(class_weight='balanced',
                                                 classes=np.unique(y_train),
                                                 y=y_train)
print(f"Class weight: {class_weights}")

y_train = to_categorical(y_train, num_label)
y_test = to_categorical(y_test, num_label)

X_train = np.expand_dims(X_train, axis=-1)
X_test = np.expand_dims(X_test, axis=-1)

Class weight: [0.99380531 0.97652174 0.98508772 0.97652174 1.02090909 0.97652174
 1.01628959 1.03502304 0.97652174 0.95982906 1.03502304 1.00717489
 1.00717489 1.02557078 1.03502304 1.03502304 0.98942731 1.00267857
 0.99380531 0.9639485 ]


## Train model

In [26]:
# Initial the SNN network

# Shape the input to right size
inputs = Input(shape=(X_train.shape[1:]))
# print(f"inputs = Input(shape={(X_train.shape[1:])})")
# print(f"core_size = {model_configs[0][1]}")

# Flatten the inputs
flattened_inputs = Flatten()(inputs)

layer_input = flattened_inputs
# For loop for each layer
for layer_ind in range(number_of_layers-1):
  layer = []
  # For each core in each layer
  for core_ind in range(model_configs[layer_ind][0]):
    core = Lambda(lambda x, start=model_configs[layer_ind][1]*core_ind, end=model_configs[layer_ind][1]*(core_ind+1): x[:, start:end])(layer_input)
    core = Tea(units=model_configs[layer_ind][2], threshold = network_activation[layer_ind][0], activation_factor = network_activation[layer_ind][1], name=f'tea_{layer_ind}_{core_ind}')(core)
    # print(f"core = Tea(units={model_configs[layer_ind][2]}, threshold = {network_activation[layer_ind][0]}, activation_factor = {network_activation[layer_ind][1]}, name=f'tea_{layer_ind}_{core_ind}')(core)")
    layer.append(core)

  layer_input = Concatenate(axis=1)(layer)
  # print(f"layer_input = Concatenate(axis=1)({layer})")

core = Tea(units=model_configs[number_of_layers-1][2], threshold=network_activation[number_of_layers-1][0], activation_factor=network_activation[number_of_layers-1][1], name=f'tea_{number_of_layers-1}')(layer_input)
# print(f"core = Tea(units={model_configs[number_of_layers-1][2]}, threshold={network_activation[number_of_layers-1][0]}, activation_factor={network_activation[number_of_layers-1][1]}, name=f'tea_{number_of_layers-1}')(layer_input)")
network = AdditivePooling(num_label)(core)

E0000 00:00:1772882850.541203    1922 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [27]:
existing_runs = [
    d for d in os.listdir(LOG_DIR)
    if d.startswith("run_")
]

run_numbers = [int(d.split("_")[1]) for d in existing_runs] if existing_runs else [0]
next_run = max(run_numbers) + 1

run_dir = os.path.join(LOG_DIR, f"run_{next_run:02d}")
checkpoint_dir = os.path.join(run_dir, "checkpoints")

os.makedirs(checkpoint_dir)

print("Run directory:", run_dir)

Run directory: /workspaces/SNN_framework/Software/log/6DMG/run_01


In [28]:
# Train
predictions = Activation('softmax')(network)

model = Model(inputs=inputs, outputs=predictions)

model.compile(loss=CategoricalCrossentropy(),
              optimizer=Adam(),
              metrics=['accuracy'],
              run_eagerly=True)

def lr_schedule(epoch):
    if epoch <= 30:
        return 0.001
    elif epoch <= 100:
        return 0.0001
    else:
        return 0.00001
reduce_lr = LearningRateScheduler(lr_schedule)

# Using callback EarlyStopping
early_stopping = EarlyStopping(
    monitor='val_accuracy',  # monitor the accuracy of validation set
    patience=100,  # Allow max 5 epoch without improvement
    min_delta=0,
    mode='max',
    verbose=1,
    restore_best_weights=True,
)

checkpoint = ModelCheckpoint(
    filepath=os.path.join(checkpoint_dir, "best_model.keras"),
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    verbose=1,
    validation_split=0.2,
    callbacks=[reduce_lr, early_stopping, checkpoint])

print(history.history.keys())
score = model.evaluate(X_test, y_test, verbose=0)

print("Validation Accuracy: ",max(history.history['val_accuracy']))
print("Test Loss: ", score[0])
print("Test Accuracy: ", score[1])

print("\n==========================")
model_path = os.path.join(run_dir, "tea_model.keras")
model.save(model_path)

print("Model saved:", model_path)

metrics = {
    "val_accuracy": float(max(history.history['val_accuracy'])),
    "test_accuracy": float(score[1]),
    "test_loss": float(score[0]),
    "epochs_trained": len(history.history["loss"]),
    "batch_size": BATCH_SIZE,
    "total_epochs": EPOCHS
}

with open(os.path.join(run_dir, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=4)

Epoch 1/300


W0000 00:00:1772882851.201120    1922 cpu_allocator_impl.cc:82] Allocation of 44466968 exceeds 10% of free system memory.


57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 305ms/step - accuracy: 0.1783 - loss: 2.8530
Epoch 1: val_accuracy improved from None to 0.42269, saving model to /workspaces/SNN_framework/Software/log/6DMG/run_01/checkpoints/best_model.keras

Epoch 1: finished saving model to /workspaces/SNN_framework/Software/log/6DMG/run_01/checkpoints/best_model.keras
57/57 ━━━━━━━━━━━━━━━━━━━━ 20s 328ms/step - accuracy: 0.3164 - loss: 2.4594 - val_accuracy: 0.4227 - val_loss: 1.8211 - learning_rate: 0.0010
Epoch 2/300
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 319ms/step - accuracy: 0.6390 - loss: 1.4807
Epoch 2: val_accuracy improved from 0.42269 to 0.70412, saving model to /workspaces/SNN_framework/Software/log/6DMG/run_01/checkpoints/best_model.keras

Epoch 2: finished saving model to /workspaces/SNN_framework/Software/log/6DMG/run_01/checkpoints/best_model.keras
57/57 ━━━━━━━━━━━━━━━━━━━━ 20s 346ms/step - accuracy: 0.7067 - loss: 1.2672 - val_accuracy: 0.7041 - val_loss: 0.9115 - learning_rate: 0.0010
Epoch 3/300
57/57 ━━━━━

In [16]:
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 238, 13,   │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 3094)      │          0 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_35 (Lambda)  │ (None, 238)       │          0 │ flatten_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_36 (Lambda)  │ (None, 238)       │          0 │ flatten_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_37 (Lambda)  │ (None, 238)       │          0 │ flatten_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_38 (Lambda)  │ (None, 238)       │          0 │ flatten_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_39 (Lambda)  │ (None, 238)       │          0 │ flatten_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_40 (Lambda)  │ (None, 238)       │          0 │ flatten_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_41 (Lambda)  │ (None, 238)       │          0 │ flatten_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_42 (Lambda)  │ (None, 238)       │          0 │ flatten_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_43 (Lambda)  │ (None, 238)       │          0 │ flatten_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_44 (Lambda)  │ (None, 238)       │          0 │ flatten_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_45 (Lambda)  │ (None, 238)       │          0 │ flatten_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_46 (Lambda)  │ (None, 238)       │          0 │ flatten_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_47 (Lambda)  │ (None, 238)       │          0 │ flatten_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_0_0 (Tea)       │ (None, 64)        │     30,528 │ lambda_35[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_0_1 (Tea)       │ (None, 64)        │     30,528 │ lambda_36[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_0_2 (Tea)       │ (None, 64)        │     30,528 │ lambda_37[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_0_3 (Tea)       │ (None, 64)        │     30,528 │ lambda_38[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_0_4 (Tea)       │ (None, 64)        │     30,528 │ lambda_39[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_0_5 (Tea)       │ (None, 64)        │     30,528 │ lambda_40[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_0_6 (Tea)       │ (None, 64)        │     30,528 │ lambda_41[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_0_7 (Tea)       │ (None, 64)        │     30,528 │ lambda_42[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tea_0_8 (Tea)       │ (None, 64)        │     30,528 │ lambda_43[0][0]   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 1,254,802 (4.79 MB)

 Trainable params: 314,032 (1.20 MB)

 Non-trainable params: 312,704 (1.19 MB)

 Optimizer params: 628,066 (2.40 MB)

In [17]:
plot_model(model, to_file=TRAIN_DIR+'/model_architecture.png', show_shapes=True, show_layer_names=True)


You must install graphviz (see instructions at https://graphviz.gitlab.io/download/) for `plot_model` to work.
